In [1]:
%%writefile double_evennumber.cu

#include <stdio.h>
#include "cuda_runtime.h"
#include "device_launch_parameters.h"

#define N 8

__global__ void doubleEven(int *a)
{
    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx < N)
    {
        if (a[idx] % 2 == 0)
        {
            a[idx] = a[idx] * 2;
        }
    }
}

int main()
{
    int host_a[N] = {1, 2, 3, 4, 5, 6, 7, 8};

    // device pointer
    int *device_a;

    // allocate memory on device
    cudaMalloc((void **)&device_a, N * sizeof(int));

    // copy input array from host to device
    cudaMemcpy(device_a, host_a, N * sizeof(int), cudaMemcpyHostToDevice);

    // launch kernel: one block, N threads (one per element)
    doubleEven<<<1, N>>>(device_a);
    cudaDeviceSynchronize();

    // copy result back from device to host
    cudaMemcpy(host_a, device_a, N * sizeof(int), cudaMemcpyDeviceToHost);

    // display the result
    printf("Result: ");
    for (int i = 0; i < N; i++)
        printf("%d ", host_a[i]);
    printf("\n");

    // free memory
    cudaFree(device_a);

    return 0;
}

Writing double_evennumber.cu


In [2]:
!nvcc double_evennumber.cu -o double_evennumber

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [3]:
!./double_evennumber

Result: 1 4 3 8 5 12 7 16 
